In [1]:
# Set up google colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Setup cell
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

%matplotlib inline
plt.rcParams["figure.figsize"] = (10.0, 8.0) # Set default size of plots
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gray"

# 1. Data Reading and Pre-processing

In [6]:
# Pre-processing Cell
# Reads data and processes it in exact same way as in arima.ipynb
# Read Weather dataset
df = pd.read_csv('/content/drive/MyDrive/mpi_roof_2025.csv', encoding = "unicode_escape")

# Selecting only 'Date Time' and 'T(degC)'
df = df[['Date Time', 'T (degC)']]

# Dropping potential NA values
df = df.dropna()

# Converting to proper date time format
df["Date Time"] = pd.to_datetime(df["Date Time"], format = "mixed")
df = df.set_index("Date Time").asfreq("10min")

# Select rows for short-term forecasting
# Total of 72 hours
df_short = df["2025-01-01":"2025-01-05"]

# Select rows for long-term forecasting
# Total of 5 months
df_long = df["2025-01-01":"2025-05-31"]

# 2. Short-term Forecasting

In [7]:
# Train/test split
# Training: 48 hours
# Testing: 24 hours
train = df_short["2025-01-01":"2025-01-04 23:50:00"]
test  = df_short["2025-01-05":"2025-01-05 23:50:00"]

train_vals = train["T (degC)"].values.reshape(-1, 1)
test_vals  = test["T (degC)"].values.reshape(-1, 1)

# Scale to [0,1]
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_vals)
test_scaled  = scaler.transform(test_vals)

# Create sequences
def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])
        y.append(data[i + seq_len])
    return torch.tensor(np.array(X), dtype=torch.float32), \
           torch.tensor(np.array(y), dtype=torch.float32)

SEQ_LEN = 144  # use last 24 hours to predict next step

X_train, y_train = create_sequences(train_scaled, SEQ_LEN)
X_test,  y_test  = create_sequences(
    np.concatenate([train_scaled[-SEQ_LEN:], test_scaled]), SEQ_LEN
)

print(f"X_train: {X_train.shape}")  # (431, 144, 1)
print(f"X_test:  {X_test.shape}")   # (144, 144, 1)

X_train: torch.Size([431, 144, 1])
X_test:  torch.Size([144, 144, 1])


In [8]:
# LSTM Architecture
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size,
                            hidden_size,
                            num_layers,
                            batch_first=True,
                            dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])  # take last timestep

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model     = LSTMModel().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using device: cpu
